# 03 · Baseline: logistic regression

**Questions this notebook answers**

1. How well does a simple, interpretable model rank fraud (PR-AUC, with uncertainty)?
2. Does class weighting help it?
3. Which inputs does it lean on?

**Protocol** (fixed in `notes/00_preregistration.md`, amendment A2): dev split only, locked
5-fold CV × 3 repeats = 15 validation folds. One-hot categoricals, `log1p` on amount and balance,
standardised numerics, L2 penalty with `C = 1`. Two pipelines are compared on the same folds with
the corrected resampled t-test; the weighted version replaces the unweighted one only if the 95% CI
of the PR-AUC difference lies entirely above 0.

> **Kaggle edition.** Generated from the project repository by `scripts/build_kaggle.py`. Attach the dataset [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) (CC0) before running. The first code cell installs the project's helper package into `/kaggle/working` (plus any results this notebook reads from earlier notebooks). Library versions on Kaggle may differ from the repo's `uv.lock`, so the last digits can differ.

In [ ]:
# Bootstrap: write the project package (and upstream artifacts) into the working
# directory. Auto-generated by scripts/build_kaggle.py.
import base64, os, sys
from pathlib import Path
WORK = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
FILES = {'src/fraud_e2e/__init__.py': 'def hello() -> str:\n    return "Hello from fraud-e2e!"\n', 'src/fraud_e2e/config.py': '"""Project-wide constants: paths, seed, column groups.\n\nPaths resolve to the repo layout locally and to the Kaggle input mount when the\ncode runs inside a Kaggle notebook, so the same notebooks work in both places.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nSEED = 42\n\n# Repo root = two levels above this file (src/fraud_e2e/config.py).\n_REPO_ROOT = Path(__file__).resolve().parents[2]\n# Kaggle mounts datasets under /kaggle/input/<slug>/ (layout varies), outputs go to\n# /kaggle/working. Env overrides exist only so the Kaggle build can be tested locally.\n_KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))\n_KAGGLE_WORKING = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))\n_RAW_NAME = "credit_card_fraud_2026.csv"\nON_KAGGLE = _KAGGLE_INPUT.is_dir() and "FRAUD_E2E_ROOT" not in os.environ\n\nif ON_KAGGLE:\n    ROOT = _KAGGLE_WORKING\n    _found = sorted(_KAGGLE_INPUT.rglob(_RAW_NAME))\n    if not _found:\n        raise FileNotFoundError(f"Attach the dataset: no {_RAW_NAME} under {_KAGGLE_INPUT}")\n    RAW_CSV = _found[0]\nelse:\n    ROOT = Path(os.environ.get("FRAUD_E2E_ROOT", _REPO_ROOT))\n    RAW_CSV = ROOT / "data" / "raw" / _RAW_NAME\n\nSPLITS_CSV = ROOT / "data" / "splits" / "splits.csv"\nREPORTS_DIR = ROOT / "reports"\nFIGURES_DIR = REPORTS_DIR / "figures"\nCONFIGS_DIR = ROOT / "configs"\n\n# SHA-256 of the canonical CSV (see data/raw/SOURCE.md). Loading fails loudly if\n# the file on disk is not byte-identical to the one the analysis was built on.\nRAW_SHA256 = "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61"\n\nID_COL = "transaction_id"\nTARGET = "is_fraud"\nAMOUNT_COL = "amount_usd"\n\nCATEGORICAL_COLS = [\n    "merchant_category",\n    "card_type",\n    "auth_method",\n    "channel",\n    "device_type",\n]\n\nBOOLEAN_COLS = [\n    "is_foreign_transaction",\n    "is_new_merchant",\n    "used_vpn",\n    "ip_country_mismatch",\n    "billing_shipping_mismatch",\n    "is_ai_generated_scam_attempt",\n]\n\n# Integer-valued columns with few distinct values (counts / clock fields).\nDISCRETE_COLS = [\n    "txn_count_last_24h",\n    "cvv_retry_count",\n    "prior_disputes",\n    "time_of_day_hour",\n    "day_of_week",\n]\n\nCONTINUOUS_COLS = [\n    "amount_usd",\n    "hours_since_last_txn",\n    "distance_from_home_km",\n    "card_age_months",\n    "customer_age",\n    "account_balance_usd",\n    "velocity_score",\n    "merchant_risk_score",\n]\n\nNUMERIC_COLS = CONTINUOUS_COLS + DISCRETE_COLS\nFEATURE_COLS = CATEGORICAL_COLS + BOOLEAN_COLS + NUMERIC_COLS\n', 'src/fraud_e2e/cost.py': '"""Business cost model for flag decisions.\n\nTwo-tier policy (primary): each scored transaction is approved or sent to\nstep-up (review / extra authentication).\n\n  TP  fraud, step-up       -> review cost\n  FP  legit, step-up       -> review cost + good-customer friction cost\n  FN  fraud, approved      -> fn_loss_fraction x amount_usd + fixed chargeback cost\n  TN  legit, approved      -> 0\n\nThree-tier policy (supplementary) adds "decline" and lets a share of fraud pass\nstep-up; see `ThreeTierParams` and `three_tier_costs`.\n\nAll values come from configs/cost.yaml. They are ILLUSTRATIVE assumptions, not\nbank data (`COST_LABEL`).\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport yaml\n\nfrom fraud_e2e import config as C\n\nCOST_LABEL = "Illustrative assumed costs (not bank data)"\n\n\n@dataclass(frozen=True)\nclass CostParams:\n    review_cost_per_alert: float  # USD per flagged transaction (TP and FP)\n    fp_friction_cost: float  # USD extra per wrongly flagged good customer\n    fn_loss_fraction: float  # multiplier on amount_usd lost on a missed fraud (0..2)\n    fn_fixed_cost: float  # USD per missed fraud (chargeback handling)\n    status: str = "proposed"  # proposed -> accepted (illustrative) / confirmed\n\n    def __post_init__(self):\n        for name in ("review_cost_per_alert", "fp_friction_cost", "fn_fixed_cost"):\n            if getattr(self, name) < 0:\n                raise ValueError(f"{name} must be >= 0")\n        if not 0 <= self.fn_loss_fraction <= 2:\n            raise ValueError("fn_loss_fraction must be within [0, 2]")\n\n    @property\n    def usable(self) -> bool:\n        return self.status in {"accepted", "confirmed"}\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n    def numeric(self) -> dict:\n        d = self.to_dict()\n        d.pop("status")\n        return d\n\n\n@dataclass(frozen=True)\nclass ThreeTierParams:\n    stepup_fraud_pass_rate: float  # share of fraud that still passes step-up (0..1)\n    decline_fp_cost: float  # USD per good customer hard-declined\n    decline_tp_cost: float  # USD per fraud declined\n    status: str = "proposed"\n\n    def __post_init__(self):\n        if not 0 <= self.stepup_fraud_pass_rate <= 1:\n            raise ValueError("stepup_fraud_pass_rate must be within [0, 1]")\n        if self.decline_fp_cost < 0 or self.decline_tp_cost < 0:\n            raise ValueError("decline costs must be >= 0")\n\n    @property\n    def usable(self) -> bool:\n        return self.status in {"accepted", "confirmed"}\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef _load_yaml(path: Path | None) -> dict:\n    path = Path(path or C.CONFIGS_DIR / "cost.yaml")\n    with open(path) as f:\n        return yaml.safe_load(f)\n\n\ndef load_cost_params(path: Path | None = None) -> CostParams:\n    raw = _load_yaml(path)\n    return CostParams(**raw["params"], status=raw.get("status", "proposed"))\n\n\ndef load_three_tier_params(path: Path | None = None) -> ThreeTierParams:\n    raw = _load_yaml(path)["three_tier"]\n    return ThreeTierParams(**raw["params"], status=raw.get("status", "proposed"))\n\n\ndef load_sensitivity_factors(path: Path | None = None) -> list[float]:\n    return list(_load_yaml(path).get("sensitivity_factors", [0.5, 2.0]))\n\n\ndef require_usable(*params) -> None:\n    for p in params:\n        if not p.usable:\n            raise PermissionError(\n                f"{type(p).__name__} status is \'{p.status}\'; it must be accepted before use"\n            )\n\n\ndef scaled(p: CostParams, name: str, factor: float) -> CostParams:\n    return replace(p, **{name: getattr(p, name) * factor})\n\n\n# --------------------------------------------------------------------------- two-tier\n\n\ndef outcome_costs(y_true, flagged, amount, p: CostParams) -> np.ndarray:\n    """Per-transaction cost in USD for a given set of flag decisions."""\n    y = np.asarray(y_true).astype(bool)\n    f = np.asarray(flagged).astype(bool)\n    a = np.asarray(amount, dtype=float)\n    cost = np.zeros(len(y))\n    cost[f] += p.review_cost_per_alert\n    cost[f & ~y] += p.fp_friction_cost\n    miss = y & ~f\n    cost[miss] += p.fn_loss_fraction * a[miss] + p.fn_fixed_cost\n    return cost\n\n\ndef total_cost(y_true, flagged, amount, p: CostParams) -> float:\n    return float(outcome_costs(y_true, flagged, amount, p).sum())\n\n\ndef reference_costs(y_true, amount, p: CostParams) -> dict:\n    """Costs of the two trivial policies every model must beat."""\n    y = np.asarray(y_true)\n    return {\n        "flag_nothing": total_cost(y, np.zeros_like(y, bool), amount, p),\n        "flag_everything": total_cost(y, np.ones_like(y, bool), amount, p),\n    }\n\n\ndef confusion_by_threshold(y_true, score, amount, thresholds) -> dict[str, np.ndarray]:\n    """TP/FP/FN/TN counts and missed-fraud USD for \'flag if score >= t\', for every t.\n\n    Cost is linear in these sufficient statistics, so they are all a front end needs to\n    recompute cost for arbitrary parameters.\n    """\n    y = np.asarray(y_true).astype(bool)\n    s = np.asarray(score, dtype=float)\n    a = np.asarray(amount, dtype=float)\n    t = np.asarray(thresholds, dtype=float)\n    order = np.argsort(s)\n    s_sorted, y_sorted, a_sorted = s[order], y[order], a[order]\n    # number of rows with score < t  (they are approved)\n    n_below = np.searchsorted(s_sorted, t, side="left")\n    cum_pos = np.r_[0, np.cumsum(y_sorted)]\n    cum_pos_amt = np.r_[0, np.cumsum(np.where(y_sorted, a_sorted, 0.0))]\n    fn = cum_pos[n_below]\n    fn_amount = cum_pos_amt[n_below]\n    tn = n_below - fn\n    tp = y.sum() - fn\n    fp = (len(y) - n_below) - tp\n    return {\n        "threshold": t,\n        "tp": tp.astype(int),\n        "fp": fp.astype(int),\n        "fn": fn.astype(int),\n        "tn": tn.astype(int),\n        "fn_amount_usd": fn_amount,\n    }\n\n\ndef cost_from_confusion(conf: dict, p: CostParams) -> np.ndarray:\n    alerts = conf["tp"] + conf["fp"]\n    return (\n        alerts * p.review_cost_per_alert\n        + conf["fp"] * p.fp_friction_cost\n        + p.fn_loss_fraction * conf["fn_amount_usd"]\n        + conf["fn"] * p.fn_fixed_cost\n    )\n\n\n# --------------------------------------------------------------------------- three-tier\n\n\ndef three_tier_costs(y_true, action, amount, p: CostParams, q: ThreeTierParams) -> np.ndarray:\n    """Per-transaction cost for actions 0 = approve, 1 = step-up, 2 = decline.\n\n    Step-up: review cost for everyone, friction for legit, and a share\n    `stepup_fraud_pass_rate` of fraud still gets through (expected FN cost).\n    Decline: `decline_fp_cost` for legit, `decline_tp_cost` for fraud.\n    """\n    y = np.asarray(y_true).astype(bool)\n    act = np.asarray(action)\n    a = np.asarray(amount, dtype=float)\n    fn_cost = p.fn_loss_fraction * a + p.fn_fixed_cost\n    cost = np.zeros(len(y))\n    approve, stepup, decline = act == 0, act == 1, act == 2\n    cost[approve & y] = fn_cost[approve & y]\n    cost[stepup] += p.review_cost_per_alert\n    cost[stepup & ~y] += p.fp_friction_cost\n    cost[stepup & y] += q.stepup_fraud_pass_rate * fn_cost[stepup & y]\n    cost[decline & ~y] = q.decline_fp_cost\n    cost[decline & y] = q.decline_tp_cost\n    return cost\n\n\ndef three_tier_actions(score, t_stepup: float, t_decline: float) -> np.ndarray:\n    s = np.asarray(score, dtype=float)\n    return np.where(s >= t_decline, 2, np.where(s >= t_stepup, 1, 0))\n', 'src/fraud_e2e/cv.py': '"""Repeated stratified CV on the dev split using the locked fold assignment."""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats as ss\nfrom sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.features import feature_frame\nfrom fraud_e2e.models import fit_pipeline, normalise_spec, spec_id\n\nREPEATS = (0, 1, 2)\n\n\ndef fold_metrics(y, p) -> dict:\n    p_clip = np.clip(p, 1e-7, 1 - 1e-7)\n    return {\n        "pr_auc": average_precision_score(y, p),\n        "roc_auc": roc_auc_score(y, p),\n        "brier": brier_score_loss(y, p),\n        "log_loss": log_loss(y, p_clip, labels=[0, 1]),\n        "base_rate": float(np.mean(y)),\n    }\n\n\n@dataclass\nclass CVResult:\n    name: str\n    spec: dict\n    folds: pd.DataFrame  # one row per (repeat, fold)\n    oof: pd.DataFrame  # one row per (repeat, transaction)\n    seconds: float = 0.0\n    extra: dict = field(default_factory=dict)\n\n    def summary(self) -> dict:\n        out = {"name": self.name, "spec_id": spec_id(self.spec), "n_folds": len(self.folds)}\n        for m in ["pr_auc", "roc_auc", "brier", "log_loss"]:\n            v = self.folds[m].to_numpy()\n            out[f"{m}_mean"] = float(v.mean())\n            out[f"{m}_sd"] = float(v.std(ddof=1))\n        out["base_rate_mean"] = float(self.folds["base_rate"].mean())\n        out["seconds"] = self.seconds\n        return out\n\n\ndef run_cv(dev: pd.DataFrame, spec: dict, name: str, repeats=REPEATS) -> CVResult:\n    """Fit on 4 folds, score the 5th, for every fold of every repeat. dev must be dev-only."""\n    if "split" in dev and not dev["split"].eq("dev").all():\n        raise ValueError("run_cv received non-dev rows; the holdout must not enter CV")\n    spec = normalise_spec(spec)\n    X = feature_frame(dev, spec["drop"])\n    y = dev[C.TARGET].to_numpy()\n    rows, oofs = [], []\n    t0 = time.perf_counter()\n    for r in repeats:\n        fold_col = dev[f"fold_r{r}"].to_numpy()\n        for k in np.unique(fold_col):\n            tr, va = fold_col != k, fold_col == k\n            model = fit_pipeline(spec, X[tr], y[tr])\n            p = model.predict_proba(X[va])[:, 1]\n            rows.append(\n                {\n                    "repeat": r,\n                    "fold": int(k),\n                    "n_train": int(tr.sum()),\n                    "n_val": int(va.sum()),\n                    "n_pos_val": int(y[va].sum()),\n                    **fold_metrics(y[va], p),\n                }\n            )\n            oofs.append(\n                pd.DataFrame(\n                    {\n                        C.ID_COL: dev.loc[va, C.ID_COL].to_numpy(),\n                        "repeat": r,\n                        "fold": int(k),\n                        "y": y[va],\n                        "amount_usd": dev.loc[va, C.AMOUNT_COL].to_numpy(),\n                        "score": p,\n                    }\n                )\n            )\n    return CVResult(\n        name=name,\n        spec=spec,\n        folds=pd.DataFrame(rows),\n        oof=pd.concat(oofs, ignore_index=True),\n        seconds=time.perf_counter() - t0,\n    )\n\n\ndef corrected_ttest(\n    a: CVResult,\n    b: CVResult,\n    metric: str = "pr_auc",\n    alpha: float = 0.05,\n    lower_is_better: bool = False,\n) -> dict:\n    """Paired comparison b - a over identical folds, Nadeau & Bengio corrected resampled t-test.\n\n    Variance of the mean difference is inflated from s^2/J to (1/J + n_test/n_train) s^2\n    to account for overlapping training sets across folds and repeats.\n    """\n    key = ["repeat", "fold"]\n    m = a.folds[key + [metric, "n_train", "n_val"]].merge(\n        b.folds[key + [metric]], on=key, suffixes=("_a", "_b"), validate="one_to_one"\n    )\n    if len(m) != len(a.folds) or len(m) != len(b.folds):\n        raise ValueError("results are not on the same folds")\n    d = (m[f"{metric}_b"] - m[f"{metric}_a"]).to_numpy()\n    J = len(d)\n    ratio = float(m["n_val"].mean() / m["n_train"].mean())\n    sd = d.std(ddof=1)\n    se = np.sqrt((1 / J + ratio) * sd**2)\n    mean = float(d.mean())\n    tcrit = ss.t.ppf(1 - alpha / 2, J - 1)\n    t = mean / se if se > 0 else np.inf * np.sign(mean)\n    p = float(2 * ss.t.sf(abs(t), J - 1)) if np.isfinite(t) else 0.0\n    lo, hi = mean - tcrit * se, mean + tcrit * se\n    return {\n        "a": a.name,\n        "b": b.name,\n        "metric": metric,\n        "mean_diff_b_minus_a": mean,\n        "ci_low": float(lo),\n        "ci_high": float(hi),\n        "p_value": p,\n        "n_folds": J,\n        "b_wins_per_folds": int((d > 0).sum()),\n        "lower_is_better": lower_is_better,\n        "verdict": _verdict(lo, hi, lower_is_better),\n    }\n\n\ndef _verdict(lo: float, hi: float, lower_is_better: bool) -> str:\n    if lower_is_better:\n        lo, hi = -hi, -lo\n    if lo > 0:\n        return "b better"\n    if hi < 0:\n        return "a better"\n    return "not distinguishable"\n\n\ndef pick(\n    simpler: CVResult, complex_: CVResult, metric: str = "pr_auc", lower_is_better: bool = False\n) -> tuple[CVResult, dict]:\n    """Pre-registered rule: keep the simpler pipeline unless the complex one is clearly better."""\n    test = corrected_ttest(simpler, complex_, metric, lower_is_better=lower_is_better)\n    better = test["verdict"] == "b better"\n    test["chosen"] = complex_.name if better else simpler.name\n    return (complex_ if better else simpler), test\n\n\ndef metric_ci(res: CVResult, metric: str = "pr_auc", alpha: float = 0.05) -> dict:\n    """Mean over folds with a 95% interval using the same corrected variance as the paired test."""\n    v = res.folds[metric].to_numpy()\n    J = len(v)\n    ratio = float(res.folds["n_val"].mean() / res.folds["n_train"].mean())\n    se = np.sqrt((1 / J + ratio) * v.var(ddof=1))\n    t = ss.t.ppf(1 - alpha / 2, J - 1)\n    return {\n        "mean": float(v.mean()),\n        "sd": float(v.std(ddof=1)),\n        "ci_low": float(v.mean() - t * se),\n        "ci_high": float(v.mean() + t * se),\n    }\n\n\nCV_DIR = C.REPORTS_DIR / "cv"\n# fixed gzip header timestamp so re-runs produce byte-identical files\nGZIP = {"method": "gzip", "mtime": 0}\n\n\ndef save_result(res: CVResult, folder=CV_DIR, with_oof: bool = True) -> None:\n    """Fold metrics + spec (always committed) and OOF predictions (gzip CSV)."""\n    folder.mkdir(parents=True, exist_ok=True)\n    res.folds.to_csv(folder / f"{res.name}.folds.csv", index=False, float_format="%.6g")\n    meta = {\n        "name": res.name,\n        "spec": res.spec,\n        "spec_id": spec_id(res.spec),\n        **res.extra,\n    }\n    (folder / f"{res.name}.spec.json").write_text(json.dumps(meta, indent=2, default=str) + "\\n")\n    if with_oof:\n        res.oof.to_csv(\n            folder / f"{res.name}.oof.csv.gz",\n            index=False,\n            float_format="%.6g",\n            compression=GZIP,\n        )\n\n\ndef load_result(name: str, folder=CV_DIR) -> CVResult:\n    meta = json.loads((folder / f"{name}.spec.json").read_text())\n    folds = pd.read_csv(folder / f"{name}.folds.csv")\n    oof_path = folder / f"{name}.oof.csv.gz"\n    oof = pd.read_csv(oof_path) if oof_path.exists() else pd.DataFrame()\n    return CVResult(name=name, spec=meta["spec"], folds=folds, oof=oof)\n\n\ndef verdict_text(test: dict) -> str:\n    """Human wording of a paired-test verdict, naming the winner."""\n    if test["verdict"] == "b better":\n        return f"`{test[\'b\']}` is better"\n    if test["verdict"] == "a better":\n        return f"`{test[\'a\']}` is better"\n    return "not distinguishable"\n', 'src/fraud_e2e/data.py': '"""Loading the raw CSV and data-quality checks.\n\nEvery check returns a plain dict so results can be dumped to JSON and quoted in\ndocuments without retyping numbers by hand.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n# Allowed domain per column. Numeric ranges are *plausibility* bounds chosen\n# before looking at the data (e.g. an hour must be 0-23); they are not fitted.\nEXPECTED_LEVELS: dict[str, set[str]] = {\n    "merchant_category": {\n        "Online Retail",\n        "Groceries",\n        "Restaurants",\n        "Electronics",\n        "Fuel",\n        "Travel",\n        "Utilities",\n        "Healthcare",\n        "Gaming",\n        "Streaming",\n        "Gift Cards",\n        "Crypto Exchange",\n    },\n    "card_type": {"Visa", "Mastercard", "Amex", "RuPay", "Discover"},\n    "auth_method": {"3D Secure", "OTP", "PIN", "Biometric", "No Authentication"},\n    "channel": {"Online", "POS", "Contactless", "In-App", "ATM"},\n    "device_type": {\n        "Android Phone",\n        "iPhone",\n        "POS Terminal",\n        "Windows PC",\n        "Mac",\n        "Tablet",\n        "Smart Watch",\n        "ATM Machine",\n    },\n}\n\nRANGES: dict[str, tuple[float, float]] = {\n    "amount_usd": (0.01, 1e6),\n    "hours_since_last_txn": (0, 24 * 365),\n    "txn_count_last_24h": (0, 500),\n    "distance_from_home_km": (0, 20_100),  # half Earth\'s circumference\n    "card_age_months": (0, 600),\n    "customer_age": (18, 110),\n    "account_balance_usd": (0, 1e8),\n    "cvv_retry_count": (0, 10),\n    "velocity_score": (0, 100),\n    "time_of_day_hour": (0, 23),\n    "day_of_week": (0, 6),\n    "merchant_risk_score": (0, 100),\n    "prior_disputes": (0, 100),\n}\n\n# Cross-column combinations that would be odd for real card traffic. They are\n# reported, not removed: the dataset is used as published.\nPLAUSIBILITY_RULES: dict[str, str] = {\n    "atm_channel_on_non_atm_device": "channel == \'ATM\' and device_type != \'ATM Machine\'",\n    "atm_device_on_non_atm_channel": "device_type == \'ATM Machine\' and channel != \'ATM\'",\n    "card_present_channel_on_desktop": (\n        "channel in [\'POS\', \'Contactless\'] and device_type in [\'Windows PC\', \'Mac\']"\n    ),\n    "shipping_mismatch_at_atm": "channel == \'ATM\' and billing_shipping_mismatch",\n    "amount_exceeds_account_balance": "amount_usd > account_balance_usd",\n}\n\n\ndef sha256_of(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_raw(path: Path | None = None, verify_hash: bool = True) -> pd.DataFrame:\n    """Read the canonical CSV with explicit dtypes."""\n    path = Path(path or C.RAW_CSV)\n    if verify_hash:\n        got = sha256_of(path)\n        if got != C.RAW_SHA256:\n            raise ValueError(f"{path} sha256 {got} != expected {C.RAW_SHA256}")\n    dtypes = {c: "category" for c in C.CATEGORICAL_COLS}\n    dtypes.update({c: "bool" for c in C.BOOLEAN_COLS})\n    df = pd.read_csv(path, dtype=dtypes)\n    df[C.TARGET] = df[C.TARGET].astype("int8")\n    return df\n\n\ndef _check(name: str, passed: bool, **detail) -> dict:\n    return {"check": name, "passed": bool(passed), **detail}\n\n\ndef quality_checks(df: pd.DataFrame) -> list[dict]:\n    """Structural checks that do not look at the label\'s relationship with features."""\n    out: list[dict] = []\n    expected_cols = [C.ID_COL, *C.FEATURE_COLS, C.TARGET]\n    out.append(\n        _check(\n            "columns_match_schema",\n            set(df.columns) == set(expected_cols),\n            missing=sorted(set(expected_cols) - set(df.columns)),\n            unexpected=sorted(set(df.columns) - set(expected_cols)),\n        )\n    )\n    out.append(_check("row_count", True, n_rows=len(df), n_cols=df.shape[1]))\n\n    n_null = df.isna().sum()\n    out.append(\n        _check(\n            "no_missing_values",\n            int(n_null.sum()) == 0,\n            missing_by_col={k: int(v) for k, v in n_null.items() if v},\n        )\n    )\n\n    ids = df[C.ID_COL]\n    out.append(_check("id_unique", ids.is_unique, n_unique=int(ids.nunique())))\n    out.append(\n        _check(\n            "id_contiguous_1_to_n",\n            bool((ids.sort_values().to_numpy() == range(1, len(df) + 1)).all()),\n        )\n    )\n\n    n_dup = int(df.drop(columns=C.ID_COL).duplicated().sum())\n    out.append(_check("no_duplicate_rows_excluding_id", n_dup == 0, n_duplicates=n_dup))\n    n_dup_x = int(df[C.FEATURE_COLS].duplicated().sum())\n    out.append(_check("no_duplicate_feature_vectors", n_dup_x == 0, n_duplicates=n_dup_x))\n\n    labels = sorted(df[C.TARGET].unique().tolist())\n    out.append(\n        _check(\n            "target_is_binary",\n            labels == [0, 1],\n            values=labels,\n            n_positive=int(df[C.TARGET].sum()),\n            positive_rate=float(df[C.TARGET].mean()),\n        )\n    )\n\n    for col, allowed in EXPECTED_LEVELS.items():\n        seen = set(df[col].astype(str).unique())\n        out.append(\n            _check(\n                f"levels_{col}",\n                seen <= allowed,\n                unexpected=sorted(seen - allowed),\n                n_levels=len(seen),\n            )\n        )\n\n    for col, (lo, hi) in RANGES.items():\n        s = df[col]\n        bad = int(((s < lo) | (s > hi)).sum())\n        out.append(\n            _check(\n                f"range_{col}",\n                bad == 0,\n                n_out_of_range=bad,\n                observed_min=float(s.min()),\n                observed_max=float(s.max()),\n                allowed=[lo, hi],\n            )\n        )\n    return out\n\n\ndef plausibility_checks(df: pd.DataFrame) -> list[dict]:\n    """Count rows matching each implausible cross-column combination."""\n    out = []\n    for name, expr in PLAUSIBILITY_RULES.items():\n        n = int(df.eval(expr).sum())\n        out.append({"rule": name, "expression": expr, "n_rows": n, "share_of_rows": n / len(df)})\n    return out\n', 'src/fraud_e2e/diagnostics.py': '"""Diagnostics that explain modelling results (not used for selection)."""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.features import feature_frame\nfrom fraud_e2e.models import build_pipeline\n\n\ndef smote_synthetic_vs_real(dev: pd.DataFrame, spec: dict, repeat: int = 0, fold: int = 0):\n    """Red-flag prevalence in real vs SMOTENC-generated fraud, for one training fold."""\n    tr = dev[dev[f"fold_r{repeat}"] != fold]\n    X, y = feature_frame(tr), tr[C.TARGET].to_numpy()\n    sampler = build_pipeline({**spec, "imbalance": "smote"}).named_steps["sampler"]\n    Xs, _ = sampler.fit_resample(X, y)\n    synthetic = Xs.iloc[len(X) :]\n    real = X[y == 1]\n    cols = C.BOOLEAN_COLS + ["cvv_retry_count", "merchant_risk_score", "velocity_score"]\n    out = pd.DataFrame(\n        {\n            "feature": cols,\n            "real_fraud_mean": [float(real[c].mean()) for c in cols],\n            "synthetic_fraud_mean": [float(synthetic[c].astype(float).mean()) for c in cols],\n            "legit_mean": [float(X.loc[y == 0, c].mean()) for c in cols],\n        }\n    )\n    out["n_real_fraud"], out["n_synthetic"] = len(real), len(synthetic)\n    return out\n', 'src/fraud_e2e/evaluate.py': '"""One-time holdout evaluation, guarded by a lock file.\n\nThe lock records exactly what was evaluated (pipeline spec id + thresholds).\nRe-running with the *same* frozen choices just reproduces the same numbers; any\nattempt to evaluate a *different* pipeline or threshold on the holdout raises,\nso the holdout cannot be used for selection by accident.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.cost import CostParams, outcome_costs, three_tier_actions, three_tier_costs\nfrom fraud_e2e.features import feature_frame\nfrom fraud_e2e.models import fit_pipeline, normalise_spec, spec_id\n\nLOCK_PATH = C.ROOT / "notes" / "holdout_lock.json"\n\n\ndef fit_final(dev: pd.DataFrame, spec: dict):\n    """Refit the chosen pipeline on all dev rows."""\n    if "split" in dev and not dev["split"].eq("dev").all():\n        raise ValueError("final model must be fit on dev rows only")\n    spec = normalise_spec(spec)\n    return fit_pipeline(spec, feature_frame(dev, spec["drop"]), dev[C.TARGET].to_numpy())\n\n\ndef _check_lock(frozen: dict, lock_path: Path, allow_create: bool) -> None:\n    if lock_path.exists():\n        locked = json.loads(lock_path.read_text())\n        if locked != frozen:\n            raise PermissionError(\n                f"Holdout already evaluated with different choices ({lock_path}). "\n                "Refusing to evaluate a new pipeline/threshold on the holdout."\n            )\n    elif allow_create:\n        lock_path.parent.mkdir(parents=True, exist_ok=True)\n        lock_path.write_text(json.dumps(frozen, indent=2) + "\\n")\n    else:\n        raise PermissionError("No holdout lock yet; pass allow_create=True for the one-time run")\n\n\ndef _metrics(y, s, amount, t_flag, p: CostParams, three_tier=None) -> dict:\n    flag = s >= t_flag\n    tp = int((flag & (y == 1)).sum())\n    fp = int((flag & (y == 0)).sum())\n    fn = int((~flag & (y == 1)).sum())\n    out = {\n        "pr_auc": average_precision_score(y, s),\n        "roc_auc": roc_auc_score(y, s),\n        "base_rate": float(y.mean()),\n        "recall": tp / (tp + fn) if tp + fn else np.nan,\n        "precision": tp / (tp + fp) if tp + fp else np.nan,\n        "alert_rate": float(flag.mean()),\n        "cost_per_1000_txn": float(outcome_costs(y, flag, amount, p).sum() / len(y) * 1000),\n        "flag_nothing_cost_per_1000_txn": float(\n            outcome_costs(y, np.zeros_like(flag), amount, p).sum() / len(y) * 1000\n        ),\n    }\n    if three_tier is not None:\n        q, t1, t2 = three_tier\n        act = three_tier_actions(s, t1, t2)\n        out["three_tier_cost_per_1000_txn"] = float(\n            three_tier_costs(y, act, amount, p, q).sum() / len(y) * 1000\n        )\n        out["three_tier_decline_rate"] = float((act == 2).mean())\n        out["three_tier_stepup_rate"] = float((act == 1).mean())\n    return out\n\n\ndef bootstrap(fn, y, s, amount, n_boot=2000, seed=C.SEED) -> dict:\n    """Stratified bootstrap (fraud and legit resampled separately) percentile 95% CIs.\n\n    `fn(y, s, amount) -> dict of scalars` is evaluated on the original sample and on each\n    resample.\n    """\n    rng = np.random.default_rng(seed)\n    y, s, amount = np.asarray(y), np.asarray(s, float), np.asarray(amount, float)\n    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)\n    point = fn(y, s, amount)\n    reps = []\n    for _ in range(n_boot):\n        idx = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]\n        reps.append(fn(y[idx], s[idx], amount[idx]))\n    reps = pd.DataFrame(reps)\n    return {\n        k: {\n            "value": float(v),\n            "ci_low": float(reps[k].quantile(0.025)),\n            "ci_high": float(reps[k].quantile(0.975)),\n        }\n        for k, v in point.items()\n    }\n\n\ndef bootstrap_metrics(y, s, amount, t_flag, p, three_tier=None, n_boot=2000, seed=C.SEED):\n    """Pre-registered holdout metrics with stratified bootstrap CIs."""\n    return bootstrap(\n        lambda yy, ss_, aa: _metrics(yy, ss_, aa, t_flag, p, three_tier), y, s, amount, n_boot, seed\n    )\n\n\ndef evaluate_holdout(\n    df_all: pd.DataFrame,\n    spec: dict,\n    t_flag: float,\n    p: CostParams,\n    three_tier=None,\n    lock_path: Path = LOCK_PATH,\n    allow_create: bool = False,\n    n_boot: int = 2000,\n) -> dict:\n    """Fit on dev, score holdout once, bootstrap CIs. three_tier = (ThreeTierParams, t1, t2)."""\n    spec = normalise_spec(spec)\n    frozen = {\n        "spec_id": spec_id(spec),\n        "spec": spec,\n        "t_flag": round(float(t_flag), 10),\n        "cost_params": p.to_dict(),\n        "three_tier": None\n        if three_tier is None\n        else {\n            "params": three_tier[0].to_dict(),\n            "t_stepup": round(float(three_tier[1]), 10),\n            "t_decline": round(float(three_tier[2]), 10),\n        },\n    }\n    _check_lock(frozen, lock_path, allow_create)\n    dev = df_all[df_all["split"] == "dev"]\n    hold = df_all[df_all["split"] == "holdout"]\n    model = fit_final(dev, spec)\n    s = model.predict_proba(feature_frame(hold, spec["drop"]))[:, 1]\n    y = hold[C.TARGET].to_numpy()\n    res = bootstrap_metrics(y, s, hold[C.AMOUNT_COL].to_numpy(), t_flag, p, three_tier, n_boot)\n    return {\n        "frozen": frozen,\n        "n_rows": len(hold),\n        "n_fraud": int(y.sum()),\n        "metrics": res,\n        "_scores": pd.DataFrame({C.ID_COL: hold[C.ID_COL].to_numpy(), "y": y, "score": s}),\n    }\n', 'src/fraud_e2e/explain.py': '"""SHAP values without the `shap` package.\n\nTree models: exact TreeSHAP from the booster itself (LightGBM `pred_contrib`,\nXGBoost `pred_contribs`). Logistic regression: exact linear SHAP,\ncoef x (x - mean(x)) on the standardised design matrix. Values are in log-odds\nof the *uncalibrated* model. One-hot columns are summed back to their source\nfeature so a categorical feature gets one value per transaction.\n"""\n\nfrom __future__ import annotations\n\nimport pandas as pd\nfrom sklearn.calibration import CalibratedClassifierCV\n\nfrom fraud_e2e.features import feature_frame, source_feature\n\n\ndef _base_pipeline(model):\n    """The fitted pipeline to explain. For a calibrated ensemble use the first member."""\n    if isinstance(model, CalibratedClassifierCV):\n        return model.calibrated_classifiers_[0].estimator\n    return model\n\n\ndef shap_values(model, df: pd.DataFrame, drop=()) -> tuple[pd.DataFrame, float]:\n    """Per-row SHAP per source feature (log-odds), and the expected value (base)."""\n    pipe = _base_pipeline(model)\n    prep, est = pipe.named_steps["prep"], pipe.named_steps["model"]\n    Xt = prep.transform(feature_frame(df, drop))\n    names = prep.get_feature_names_out()\n    kind = type(est).__name__\n    if kind == "LGBMClassifier":\n        contrib = est.booster_.predict(Xt, pred_contrib=True)\n        phi, base = contrib[:, :-1], float(contrib[0, -1])\n    elif kind == "XGBClassifier":\n        import xgboost as xgb\n\n        contrib = est.get_booster().predict(xgb.DMatrix(Xt, feature_names=None), pred_contribs=True)\n        phi, base = contrib[:, :-1], float(contrib[0, -1])\n    elif kind == "LogisticRegression":\n        mu = Xt.mean(axis=0)\n        phi = (Xt - mu) * est.coef_[0]\n        base = float(est.intercept_[0] + mu @ est.coef_[0])\n    else:\n        raise TypeError(f"no SHAP implementation for {kind}")\n    by_col = pd.DataFrame(phi, columns=names, index=df.index)\n    groups = [source_feature(n, drop) for n in names]\n    return by_col.T.groupby(groups, sort=False).sum().T, base\n\n\ndef global_importance(phi: pd.DataFrame) -> pd.DataFrame:\n    imp = phi.abs().mean().sort_values(ascending=False)\n    return imp.rename("mean_abs_shap").reset_index().rename(columns={"index": "feature"})\n', 'src/fraud_e2e/features.py': '"""Feature matrix and preprocessing (fit inside each training fold by the pipeline)."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler\n\nfrom fraud_e2e import config as C\n\nLOG_COLS = ["amount_usd", "account_balance_usd"]  # heavy right tails (P1)\n\n\ndef feature_list(drop: Sequence[str] = ()) -> list[str]:\n    unknown = set(drop) - set(C.FEATURE_COLS)\n    if unknown:\n        raise ValueError(f"Unknown features to drop: {sorted(unknown)}")\n    return [c for c in C.FEATURE_COLS if c not in set(drop)]\n\n\ndef feature_frame(df: pd.DataFrame, drop: Sequence[str] = ()) -> pd.DataFrame:\n    """Model input: feature columns only (never the ID or target), booleans as 0/1."""\n    cols = feature_list(drop)\n    X = df[cols].copy()\n    for c in C.BOOLEAN_COLS:\n        if c in X:\n            X[c] = X[c].astype(np.int8)\n    return X\n\n\ndef make_preprocessor(kind: str, drop: Sequence[str] = ()) -> ColumnTransformer:\n    """\'linear\': one-hot + log1p(heavy tails) + standardise; \'tree\': one-hot, raw numerics."""\n    cols = feature_list(drop)\n    cats = [c for c in C.CATEGORICAL_COLS if c in cols]\n    bools = [c for c in C.BOOLEAN_COLS if c in cols]\n    nums = [c for c in C.NUMERIC_COLS if c in cols]\n    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)\n    if kind == "linear":\n        logs = [c for c in nums if c in LOG_COLS]\n        plain = [c for c in nums if c not in LOG_COLS]\n        transformers = [\n            ("cat", onehot, cats),\n            ("bool", "passthrough", bools),\n            (\n                "log",\n                make_pipeline(\n                    FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()\n                ),\n                logs,\n            ),\n            ("num", StandardScaler(), plain),\n        ]\n    elif kind == "tree":\n        transformers = [\n            ("cat", onehot, cats),\n            ("bool", "passthrough", bools),\n            ("num", "passthrough", nums),\n        ]\n    else:\n        raise ValueError(f"unknown preprocessor kind {kind!r}")\n    return ColumnTransformer(transformers, verbose_feature_names_out=True)\n\n\ndef source_feature(transformed_name: str, drop: Sequence[str] = ()) -> str:\n    """Map an output column (e.g. \'cat__auth_method_PIN\') back to its input feature."""\n    _, _, rest = transformed_name.partition("__")\n    if rest in C.FEATURE_COLS:\n        return rest\n    for c in sorted(feature_list(drop), key=len, reverse=True):\n        if rest.startswith(c + "_"):\n            return c\n    raise KeyError(transformed_name)\n', 'src/fraud_e2e/models.py': '"""Model pipelines described by a plain-dict spec, so every experiment is loggable.\n\nspec = {\n    "model": "logreg" | "lgbm" | "xgb",\n    "params": {...},               # overrides of the defaults below\n    "imbalance": "none" | "class_weight" | "undersample" | "smote",\n    "drop": [...],                 # features to leave out (ablation)\n    "calibration": None | "sigmoid" | "isotonic",\n}\nSamplers sit inside an imblearn Pipeline, so they only ever see training rows.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\n\nfrom imblearn.over_sampling import SMOTENC\nfrom imblearn.pipeline import Pipeline\nfrom imblearn.under_sampling import RandomUnderSampler\nfrom lightgbm import LGBMClassifier\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.linear_model import LogisticRegression\nfrom xgboost import XGBClassifier\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.features import feature_list, make_preprocessor\n\nN_JOBS = 4  # keep the Mac responsive for other projects\n\nDEFAULTS: dict[str, dict] = {\n    "logreg": {"C": 1.0, "max_iter": 2000},\n    "lgbm": {\n        "n_estimators": 300,\n        "learning_rate": 0.05,\n        "num_leaves": 15,\n        "min_child_samples": 20,\n        "subsample": 0.8,\n        "subsample_freq": 1,\n        "colsample_bytree": 0.8,\n        "reg_lambda": 0.0,\n    },\n    "xgb": {\n        "n_estimators": 300,\n        "learning_rate": 0.05,\n        "max_depth": 4,\n        "min_child_weight": 20 * 0.0169 * (1 - 0.0169),  # ~20 rows\' worth of hessian\n        "subsample": 0.8,\n        "colsample_bytree": 0.8,\n        "reg_lambda": 1.0,\n    },\n}\n\nUNDERSAMPLE_RATIO = 0.1  # minority / majority after undersampling (legit 10:1)\nSMOTE_RATIO = 0.1  # minority / majority after SMOTENC\nSMOTE_K = 5\n\n\ndef spec_id(spec: dict) -> str:\n    """Short stable hash of a spec (for logs and the holdout lock)."""\n    blob = json.dumps(spec, sort_keys=True, default=str).encode()\n    return hashlib.sha256(blob).hexdigest()[:12]\n\n\ndef normalise_spec(spec: dict) -> dict:\n    s = {"params": {}, "imbalance": "none", "drop": [], "calibration": None, **copy.deepcopy(spec)}\n    s["drop"] = sorted(s["drop"])\n    return s\n\n\ndef _estimator(spec: dict):\n    name = spec["model"]\n    params = {**DEFAULTS[name], **spec["params"]}\n    weighted = spec["imbalance"] == "class_weight"\n    if name == "logreg":\n        return LogisticRegression(\n            **params, class_weight="balanced" if weighted else None, random_state=C.SEED\n        )\n    if name == "lgbm":\n        return LGBMClassifier(\n            **params,\n            class_weight="balanced" if weighted else None,\n            random_state=C.SEED,\n            n_jobs=N_JOBS,\n            deterministic=True,\n            force_row_wise=True,\n            verbose=-1,\n        )\n    if name == "xgb":\n        # scale_pos_weight is set from the training fold by fit_pipeline when weighted\n        return XGBClassifier(\n            **params,\n            tree_method="hist",\n            random_state=C.SEED,\n            n_jobs=N_JOBS,\n            eval_metric="logloss",\n        )\n    raise ValueError(f"unknown model {name!r}")\n\n\ndef build_pipeline(spec: dict):\n    spec = normalise_spec(spec)\n    kind = "linear" if spec["model"] == "logreg" else "tree"\n    steps = []\n    if spec["imbalance"] == "undersample":\n        steps.append(\n            (\n                "sampler",\n                RandomUnderSampler(sampling_strategy=UNDERSAMPLE_RATIO, random_state=C.SEED),\n            )\n        )\n    elif spec["imbalance"] == "smote":\n        cols = feature_list(spec["drop"])\n        cat_like = [c for c in cols if c in C.CATEGORICAL_COLS + C.BOOLEAN_COLS]\n        steps.append(\n            (\n                "sampler",\n                SMOTENC(\n                    categorical_features=cat_like,\n                    sampling_strategy=SMOTE_RATIO,\n                    k_neighbors=SMOTE_K,\n                    random_state=C.SEED,\n                ),\n            )\n        )\n    elif spec["imbalance"] not in {"none", "class_weight"}:\n        raise ValueError(f"unknown imbalance strategy {spec[\'imbalance\']!r}")\n    steps += [("prep", make_preprocessor(kind, spec["drop"])), ("model", _estimator(spec))]\n    pipe = Pipeline(steps)\n    if spec["calibration"]:\n        return CalibratedClassifierCV(pipe, method=spec["calibration"], cv=5, ensemble=True)\n    return pipe\n\n\ndef fit_pipeline(spec: dict, X, y):\n    """Build and fit; handles XGBoost\'s class weight, which needs the fold\'s class ratio."""\n    spec = normalise_spec(spec)\n    pipe = build_pipeline(spec)\n    if spec["model"] == "xgb" and spec["imbalance"] == "class_weight":\n        ratio = float((y == 0).sum() / (y == 1).sum())\n        key = (\n            "estimator__model__scale_pos_weight"\n            if spec["calibration"]\n            else "model__scale_pos_weight"\n        )\n        pipe.set_params(**{key: ratio})\n    return pipe.fit(X, y)\n', 'src/fraud_e2e/plots.py': '"""Matplotlib helpers with one consistent, readable style.\n\nColour roles (fixed, never cycled): legit = blue, fraud = orange, reference\nlines = neutral grey. Every plot sets axis labels with units.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\nLEGIT = "#2a78d6"\nFRAUD = "#eb6834"\nNEUTRAL = "#8a8985"\nINK = "#0b0b0b"\nINK_2 = "#52514e"\nGRID = "#e4e3df"\n\n\ndef set_style() -> None:\n    plt.rcParams.update(\n        {\n            "figure.dpi": 110,\n            "savefig.dpi": 150,\n            "savefig.bbox": "tight",\n            "figure.facecolor": "#fcfcfb",\n            "axes.facecolor": "#fcfcfb",\n            "axes.edgecolor": GRID,\n            "axes.labelcolor": INK_2,\n            "axes.titlecolor": INK,\n            "axes.titlesize": 12,\n            "axes.titleweight": "bold",\n            "axes.labelsize": 10,\n            "axes.spines.top": False,\n            "axes.spines.right": False,\n            "axes.grid": True,\n            "grid.color": GRID,\n            "grid.linewidth": 0.8,\n            "xtick.color": INK_2,\n            "ytick.color": INK_2,\n            "legend.frameon": False,\n            "font.size": 10,\n        }\n    )\n\n\ndef save(fig, name: str, folder: Path = C.FIGURES_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.png"\n    fig.savefig(path)\n    return path\n\n\ndef rate_bars(table: pd.DataFrame, base_rate: float, ax=None, title: str | None = None):\n    """Horizontal bars of fraud rate per level with 95% Wilson CI whiskers."""\n    t = table.sort_values("fraud_rate")\n    ax = ax or plt.subplots(figsize=(6, 0.35 * len(t) + 1.2))[1]\n    y = np.arange(len(t))\n    ax.barh(y, t["fraud_rate"] * 100, color=FRAUD, height=0.6, zorder=2)\n    err = [(t["fraud_rate"] - t["ci_low"]) * 100, (t["ci_high"] - t["fraud_rate"]) * 100]\n    ax.errorbar(\n        t["fraud_rate"] * 100,\n        y,\n        xerr=err,\n        fmt="none",\n        ecolor=INK_2,\n        elinewidth=1,\n        capsize=2,\n        zorder=3,\n    )\n    ax.axvline(base_rate * 100, color=NEUTRAL, ls="--", lw=1, zorder=1)\n    ax.set_yticks(y, [f"{lvl}  (n={n:,})" for lvl, n in zip(t["level"], t["n"])])\n    ax.set_xlabel("Fraud rate (%) — bars: 95% Wilson CI, dashed: overall rate")\n    ax.grid(axis="y", visible=False)\n    ax.set_title(title or t["feature"].iloc[0])\n    return ax\n\n\ndef binned_rate_line(\n    table: pd.DataFrame, base_rate: float, xlabel: str, ax=None, logx: bool = False\n):\n    """Fraud rate across quantile bins of a numeric feature, CI as a band."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    x = table["bin_mid"].astype(float)\n    ax.fill_between(x, table["ci_low"] * 100, table["ci_high"] * 100, color=FRAUD, alpha=0.18, lw=0)\n    ax.plot(x, table["fraud_rate"] * 100, color=FRAUD, lw=2, marker="o", ms=4)\n    ax.axhline(base_rate * 100, color=NEUTRAL, ls="--", lw=1)\n    if logx:\n        ax.set_xscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("Fraud rate (%)")\n    ax.set_ylim(bottom=0)\n    return ax\n\n\ndef class_hist(\n    df: pd.DataFrame,\n    col: str,\n    xlabel: str,\n    ax=None,\n    bins: int = 40,\n    logx: bool = False,\n    target: str = C.TARGET,\n):\n    """Overlaid histograms for legit vs fraud, each class normalised to 100%."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    x = df[col].astype(float)\n    if logx:\n        edges = np.geomspace(max(x.min(), 1e-2), x.max(), bins + 1)\n    else:\n        edges = np.histogram_bin_edges(x, bins=bins)\n    for label, colour, name in [(0, LEGIT, "Legit"), (1, FRAUD, "Fraud")]:\n        xs = x[df[target] == label]\n        ax.hist(\n            xs,\n            bins=edges,\n            weights=np.full(len(xs), 100 / len(xs)),\n            histtype="step",\n            lw=2,\n            color=colour,\n            label=f"{name} (n={len(xs):,})",\n        )\n    if logx:\n        ax.set_xscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("Share of class in bin (%)")\n    ax.legend()\n    return ax\n\n\ndef auc_forest(table: pd.DataFrame, ax=None):\n    """Univariate ROC AUC per feature with bootstrap CI; 0.5 = no signal."""\n    t = table.sort_values("separation")\n    ax = ax or plt.subplots(figsize=(6, 0.33 * len(t) + 1.2))[1]\n    y = np.arange(len(t))\n    colours = [FRAUD if a > 0.5 else LEGIT for a in t["auc"]]\n    ax.hlines(y, t["auc_ci_low"], t["auc_ci_high"], color=INK_2, lw=1.5)\n    ax.scatter(t["auc"], y, color=colours, s=40, zorder=3, edgecolor="#fcfcfb", lw=1.5)\n    ax.axvline(0.5, color=NEUTRAL, ls="--", lw=1)\n    ax.set_yticks(y, t["feature"])\n    ax.set_xlabel("Univariate ROC AUC (95% bootstrap CI); >0.5 orange = higher value → more fraud")\n    ax.grid(axis="y", visible=False)\n    return ax\n\n\nSERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]\n\n\ndef fold_dots(results: list, metric: str = "pr_auc", ax=None, xlabel: str | None = None):\n    """Per-fold metric for each pipeline (dots), with mean and corrected 95% CI (bar)."""\n    from fraud_e2e.cv import metric_ci\n\n    ax = ax or plt.subplots(figsize=(7, 0.55 * len(results) + 1.3))[1]\n    rng = np.random.default_rng(0)\n    for i, res in enumerate(results):\n        v = res.folds[metric].to_numpy()\n        ci = metric_ci(res, metric)\n        y = len(results) - 1 - i\n        ax.scatter(\n            v,\n            y + rng.uniform(-0.12, 0.12, len(v)),\n            s=14,\n            color=SERIES[0],\n            alpha=0.45,\n            lw=0,\n            zorder=2,\n        )\n        ax.hlines(y, ci["ci_low"], ci["ci_high"], color=INK, lw=2, zorder=3)\n        ax.scatter([ci["mean"]], [y], s=60, color=FRAUD, zorder=4, edgecolor="#fcfcfb", lw=1.5)\n    ax.set_yticks(range(len(results)), [r.name for r in results][::-1])\n    ax.set_xlabel(xlabel or f"{metric} per validation fold (dots); mean ± corrected 95% CI")\n    ax.grid(axis="y", visible=False)\n    return ax\n\n\ndef pr_curves(results: list, ax=None, repeat: int = 0):\n    """Precision-recall curves of OOF predictions (one repeat), with the no-skill line."""\n    from sklearn.metrics import precision_recall_curve\n\n    ax = ax or plt.subplots(figsize=(5.5, 4))[1]\n    base = None\n    for i, res in enumerate(results):\n        o = res.oof[res.oof["repeat"] == repeat]\n        prec, rec, _ = precision_recall_curve(o["y"], o["score"])\n        ax.plot(rec * 100, prec * 100, color=SERIES[i % len(SERIES)], lw=2, label=res.name)\n        base = o["y"].mean()\n    ax.axhline(base * 100, color=NEUTRAL, ls="--", lw=1, label=f"No skill ({100 * base:.2f}%)")\n    ax.set_xlabel("Recall (% of fraud caught)")\n    ax.set_ylabel("Precision (% of alerts that are fraud)")\n    ax.set_xlim(0, 100)\n    ax.set_ylim(0, 100)\n    ax.legend(loc="upper right")\n    return ax\n', 'src/fraud_e2e/reporting.py': '"""Write analysis results as committed artifacts (JSON / CSV).\n\nDocuments and the website quote numbers only from these files, so every\nartifact carries the data hash and seed it was produced with.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n\ndef _default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, np.ndarray):\n        return o.tolist()\n    if isinstance(o, pd.DataFrame):\n        return o.to_dict(orient="records")\n    raise TypeError(f"Not JSON serialisable: {type(o)}")\n\n\ndef write_json(payload: dict, name: str, folder: Path = C.REPORTS_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.json"\n    body = {"_meta": {"data_sha256": C.RAW_SHA256, "seed": C.SEED}, **payload}\n    path.write_text(json.dumps(body, indent=2, default=_default, sort_keys=False) + "\\n")\n    return path\n\n\ndef write_csv(df: pd.DataFrame, name: str, folder: Path = C.REPORTS_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.csv"\n    df.to_csv(path, index=False, float_format="%.6g")\n    return path\n\n\ndef read_json(name: str, folder: Path = C.REPORTS_DIR) -> dict:\n    return json.loads((folder / f"{name}.json").read_text())\n', 'src/fraud_e2e/splits.py': '"""Locked data splits.\n\nDesign (fixed before any label-based EDA, see notes/00_preregistration.md):\n  * 20% stratified holdout, touched once at the very end of modelling.\n  * The remaining 80% ("dev") gets repeated stratified 5-fold CV assignments\n    (3 repeats, different shuffles) so fold-to-fold variance can be reported.\nThe assignments are written to data/splits/splits.csv and committed; every\nlater stage reads that file instead of re-splitting.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nfrom fraud_e2e import config as C\n\nHOLDOUT_FRACTION = 0.20\nN_FOLDS = 5\nN_REPEATS = 3\n\n\ndef make_splits(df: pd.DataFrame, seed: int = C.SEED) -> pd.DataFrame:\n    """Return one row per transaction: split (\'dev\'/\'holdout\') and fold_r{k} for dev rows."""\n    ids = df[C.ID_COL].to_numpy()\n    y = df[C.TARGET].to_numpy()\n    dev_ids, _ = train_test_split(ids, test_size=HOLDOUT_FRACTION, stratify=y, random_state=seed)\n    out = pd.DataFrame({C.ID_COL: ids, "split": "holdout"})\n    out.loc[out[C.ID_COL].isin(dev_ids), "split"] = "dev"\n\n    dev_mask = out["split"].eq("dev").to_numpy()\n    y_dev = y[dev_mask]\n    for r in range(N_REPEATS):\n        fold = np.full(len(out), -1, dtype=int)\n        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed + r)\n        dev_fold = np.empty(dev_mask.sum(), dtype=int)\n        for k, (_, val_idx) in enumerate(skf.split(np.zeros(len(y_dev)), y_dev)):\n            dev_fold[val_idx] = k\n        fold[dev_mask] = dev_fold\n        out[f"fold_r{r}"] = fold\n    return out\n\n\ndef save_splits(splits: pd.DataFrame, path: Path = C.SPLITS_CSV) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    splits.to_csv(path, index=False)\n\n\ndef load_splits(path: Path = C.SPLITS_CSV) -> pd.DataFrame:\n    return pd.read_csv(path)\n\n\ndef attach_splits(df: pd.DataFrame, splits: pd.DataFrame | None = None) -> pd.DataFrame:\n    splits = load_splits() if splits is None else splits\n    merged = df.merge(splits, on=C.ID_COL, how="left", validate="one_to_one")\n    if merged["split"].isna().any():\n        raise ValueError("Some transactions have no split assignment")\n    return merged\n\n\ndef dev_only(df: pd.DataFrame) -> pd.DataFrame:\n    """The only slice that label-based EDA and model selection may look at."""\n    if "split" not in df.columns:\n        df = attach_splits(df)\n    return df[df["split"].eq("dev")].reset_index(drop=True)\n', 'src/fraud_e2e/stats.py': '"""Small statistics helpers for a rare-event target.\n\nWith only a few hundred positives, every rate and score here is reported with\nan interval. Intervals are Wilson (for proportions) or stratified bootstrap\npercentile intervals (for AUC), both seeded for reproducibility.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats as ss\n\nfrom fraud_e2e import config as C\n\n\ndef wilson_ci(k: np.ndarray | int, n: np.ndarray | int, alpha: float = 0.05):\n    """Wilson score interval for k successes out of n. Vectorised."""\n    k = np.asarray(k, dtype=float)\n    n = np.asarray(n, dtype=float)\n    z = ss.norm.ppf(1 - alpha / 2)\n    p = np.divide(k, n, out=np.zeros_like(k), where=n > 0)\n    denom = 1 + z**2 / n\n    centre = (p + z**2 / (2 * n)) / denom\n    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom\n    lo = np.where(k == 0, 0.0, np.clip(centre - half, 0, 1))  # exact bounds at the edges\n    hi = np.where(k == n, 1.0, np.clip(centre + half, 0, 1))\n    return lo, hi\n\n\ndef rate_table(df: pd.DataFrame, col: str, target: str = C.TARGET) -> pd.DataFrame:\n    """Fraud rate per level of `col` with 95% Wilson CI and lift vs the overall rate."""\n    g = df.groupby(col, observed=True)[target].agg(n="size", n_fraud="sum").reset_index()\n    g["fraud_rate"] = g["n_fraud"] / g["n"]\n    g["ci_low"], g["ci_high"] = wilson_ci(g["n_fraud"].to_numpy(), g["n"].to_numpy())\n    base = df[target].mean()\n    g["lift"] = g["fraud_rate"] / base\n    g.insert(0, "feature", col)\n    g = g.rename(columns={col: "level"})\n    g["level"] = g["level"].astype(str)\n    return g.sort_values("fraud_rate", ascending=False).reset_index(drop=True)\n\n\ndef chi2_association(df: pd.DataFrame, col: str, target: str = C.TARGET) -> dict:\n    """Chi-square test of independence + Cramér\'s V between a discrete column and the target."""\n    table = pd.crosstab(df[col], df[target])\n    chi2, p, dof, _ = ss.chi2_contingency(table, correction=False)\n    n = table.to_numpy().sum()\n    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))\n    return {\n        "feature": col,\n        "chi2": float(chi2),\n        "dof": int(dof),\n        "p_value": float(p),\n        "cramers_v": float(v),\n        "n_levels": int(table.shape[0]),\n    }\n\n\ndef rank_auc(x: np.ndarray, y: np.ndarray) -> float:\n    """ROC AUC of score x for binary labels y via the Mann-Whitney U statistic (ties averaged)."""\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y).astype(bool)\n    n_pos, n_neg = y.sum(), (~y).sum()\n    ranks = ss.rankdata(x)\n    return float((ranks[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))\n\n\ndef bootstrap_auc_ci(\n    x: np.ndarray, y: np.ndarray, n_boot: int = 1000, seed: int = C.SEED, alpha: float = 0.05\n) -> tuple[float, float]:\n    """Stratified bootstrap percentile CI for rank_auc.\n\n    Positives and negatives are resampled separately so every replicate keeps the class counts.\n    """\n    rng = np.random.default_rng(seed)\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y).astype(bool)\n    pos, neg = x[y], x[~y]\n    lab = np.r_[np.ones(len(pos), bool), np.zeros(len(neg), bool)]\n    aucs = np.empty(n_boot)\n    for b in range(n_boot):\n        xb = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]\n        aucs[b] = rank_auc(xb, lab)\n    return float(np.quantile(aucs, alpha / 2)), float(np.quantile(aucs, 1 - alpha / 2))\n\n\ndef univariate_numeric(\n    df: pd.DataFrame,\n    cols: list[str],\n    target: str = C.TARGET,\n    n_boot: int = 1000,\n    seed: int = C.SEED,\n) -> pd.DataFrame:\n    """Per numeric column: class medians/IQR and univariate ROC AUC with bootstrap CI.\n\n    AUC > 0.5 means larger values are associated with fraud; AUC < 0.5 the opposite.\n    `separation` = |AUC - 0.5| ranks features regardless of direction.\n    """\n    y = df[target].to_numpy()\n    rows = []\n    for i, c in enumerate(cols):\n        x = df[c].to_numpy(dtype=float)\n        auc = rank_auc(x, y)\n        lo, hi = bootstrap_auc_ci(x, y, n_boot=n_boot, seed=seed + i)\n        q = lambda a, p: float(np.quantile(a, p))  # noqa: E731\n        xp, xn = x[y == 1], x[y == 0]\n        rows.append(\n            {\n                "feature": c,\n                "median_fraud": q(xp, 0.5),\n                "iqr_fraud": [q(xp, 0.25), q(xp, 0.75)],\n                "median_legit": q(xn, 0.5),\n                "iqr_legit": [q(xn, 0.25), q(xn, 0.75)],\n                "auc": auc,\n                "auc_ci_low": lo,\n                "auc_ci_high": hi,\n                "separation": abs(auc - 0.5),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("separation", ascending=False).reset_index(drop=True)\n\n\ndef binned_rate(\n    df: pd.DataFrame, col: str, n_bins: int = 10, target: str = C.TARGET\n) -> pd.DataFrame:\n    """Fraud rate by quantile bin of a numeric column, with Wilson CI.\n\n    Low-cardinality integer columns (<= 15 distinct values) are grouped by exact value\n    instead, with the top 1% folded into the 99th-percentile value so no bin is tiny.\n    """\n    if df[col].nunique() <= 15:\n        cap = df[col].quantile(0.99)\n        x = df[col].clip(upper=cap)\n        g = df.groupby(x)[target].agg(n="size", n_fraud="sum").reset_index()\n        g["bin_mid"] = g["bin_left"] = g["bin_right"] = g[col].astype(float)\n    else:\n        bins = pd.qcut(df[col], q=n_bins, duplicates="drop")\n        g = df.groupby(bins, observed=True)[target].agg(n="size", n_fraud="sum").reset_index()\n        g["bin_mid"] = [iv.mid for iv in g[col]]\n        g["bin_left"] = [iv.left for iv in g[col]]\n        g["bin_right"] = [iv.right for iv in g[col]]\n    g["fraud_rate"] = g["n_fraud"] / g["n"]\n    g["ci_low"], g["ci_high"] = wilson_ci(g["n_fraud"].to_numpy(), g["n"].to_numpy())\n    g.insert(0, "feature", col)\n    return g.drop(columns=col)\n', 'src/fraud_e2e/summary.py': '"""One place that turns committed artifacts (reports/*) into formatted numbers.\n\nREADME, model card, interview notes and notebook 00 all quote numbers through\n`build_context()`, so no document ever contains a hand-typed result.\nTemplates use `{{key}}` placeholders; `render()` fails on any unknown key.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport re\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n\ndef _j(name: str) -> dict:\n    return json.loads((C.REPORTS_DIR / f"{name}.json").read_text())\n\n\ndef _csv(name: str) -> pd.DataFrame:\n    return pd.read_csv(C.REPORTS_DIR / f"{name}.csv")\n\n\ndef f3(x: float) -> str:\n    return f"{x:.3f}"\n\n\ndef pct(x: float, nd: int = 1) -> str:\n    return f"{100 * x:.{nd}f}%"\n\n\ndef signed(x: float, nd: int = 3) -> str:\n    return f"{x:+.{nd}f}"\n\n\ndef ci3(lo: float, hi: float) -> str:\n    return f"[{lo:.3f}, {hi:.3f}]"\n\n\ndef ci_pct(lo: float, hi: float, nd: int = 1) -> str:\n    return f"[{100 * lo:.{nd}f}%, {100 * hi:.{nd}f}%]"\n\n\ndef ci_signed(lo: float, hi: float, nd: int = 3) -> str:\n    return f"[{lo:+.{nd}f}, {hi:+.{nd}f}]"\n\n\ndef usd(x: float) -> str:\n    return f"{x:,.0f} USD"\n\n\ndef _logit(p: float) -> float:\n    return math.log(p / (1 - p))\n\n\ndef build_context() -> dict[str, str]:\n    c: dict[str, str] = {}\n    p1, eda = _j("p1_data_understanding"), _j("p1_eda_summary")\n    allrow = next(r for r in p1["class_balance"] if r["subset"] == "all")\n    dev = next(r for r in p1["class_balance"] if r["subset"] == "dev")\n    hold_bal = next(r for r in p1["class_balance"] if r["subset"] == "holdout")\n    c |= {\n        "n_rows": f"{allrow[\'n\']:,}",\n        "n_fraud": f"{allrow[\'n_fraud\']}",\n        "fraud_rate": f"{allrow[\'fraud_rate_%\']:.2f}%",\n        "fraud_rate_ci": f"[{allrow[\'ci95_low_%\']:.2f}%, {allrow[\'ci95_high_%\']:.2f}%]",\n        "legit_per_fraud": f"{(allrow[\'n\'] - allrow[\'n_fraud\']) / allrow[\'n_fraud\']:.0f}",\n        "n_features": str(sum(len(v) for v in p1["feature_groups"].values())),\n        "dev_n": f"{dev[\'n\']:,}",\n        "dev_fraud": f"{dev[\'n_fraud\']}",\n        "holdout_n": f"{hold_bal[\'n\']:,}",\n        "holdout_fraud": f"{hold_bal[\'n_fraud\']}",\n        "n_checks": f"{p1[\'n_quality_checks\']}",\n        "n_checks_failed": f"{p1[\'n_quality_checks_failed\']}",\n        "chan_dev_p": f"{p1[\'channel_device_independence\'][\'chi2_p_value\']:.2f}",\n        "chan_dev_gap_pp": f"{p1[\'channel_device_independence\'][\'max_share_gap_pp\']:.1f}",\n    }\n    worst = max(p1["plausibility_checks"], key=lambda r: r["n_rows"])\n    c |= {"implausible_rule": worst["rule"], "implausible_share": pct(worst["share_of_rows"])}\n\n    ls = eda["leakage_screen"]\n    c |= {"max_uni_auc": f3(ls["max_direction_free_auc"]), "max_uni_feature": ls["max_feature"]}\n    c["ai_flag_fraud_rate"] = pct(eda["ai_scam_flag"]["fraud_rate"])\n    rf = eda["red_flags"]\n    c |= {\n        "flags0_rate": pct(rf[0]["fraud_rate"], 2),\n        "flags_top_level": str(rf[-1]["level"]),\n        "flags_top_rate": pct(rf[-1]["fraud_rate"]),\n        "flags_top_ci": ci_pct(rf[-1]["ci_low"], rf[-1]["ci_high"]),\n    }\n    logits = [_logit(r["fraud_rate"]) for r in rf]\n    steps = [b - a for a, b in zip(logits, logits[1:], strict=False)]\n    c |= {\n        "flag_logodds_steps": ", ".join(f"{s:+.2f}" for s in steps),\n        "flag_odds_multipliers": ", ".join(f"×{math.exp(s):.1f}" for s in steps),\n    }\n    amt = eda["amount_at_risk"]\n    c |= {\n        "top10_fraud_usd_share": pct(amt["top10pct_share_of_fraud_usd"], 0),\n        "fraud_min_usd": f"{amt[\'fraud_min_usd\']:.2f} USD",\n        "legit_below_fraud_min": pct(amt["legit_share_below_fraud_min"]),\n        "max_spearman": f"{eda[\'max_abs_spearman\'][\'value\']:.2f}",\n        "max_spearman_pair": " / ".join(f"`{x}`" for x in eda["max_abs_spearman"]["pair"]),\n    }\n\n    mod = _j("p2_modeling")\n    fin = mod["final_pipeline"]\n    dec = {d["step"]: d for d in mod["decisions"]}\n    web = _j("p2_threshold_web")\n    noskill = web["n_fraud"] / web["n_rows"]\n    c |= {\n        "final_model": "logistic regression"\n        if fin["spec"]["model"] == "logreg"\n        else fin["spec"]["model"],\n        "final_spec_id": fin["spec_id"],\n        "cv_pr_auc": f3(fin["pr_auc"]["mean"]),\n        "cv_pr_auc_ci": ci3(fin["pr_auc"]["ci_low"], fin["pr_auc"]["ci_high"]),\n        "cv_pr_auc_sd": f3(fin["pr_auc"]["sd"]),\n        "cv_roc_auc": f3(fin["roc_auc"]["mean"]),\n        "cv_roc_auc_ci": ci3(fin["roc_auc"]["ci_low"], fin["roc_auc"]["ci_high"]),\n        "noskill": f"{noskill:.4f}",\n        "cv_lift": f"{fin[\'pr_auc\'][\'mean\'] / noskill:.0f}",\n    }\n\n    def d(step: str, key: str):\n        x = dec[step]\n        c[f"{key}_diff"] = signed(x["mean_diff_b_minus_a"])\n        c[f"{key}_ci"] = ci_signed(x["ci_low"], x["ci_high"])\n        c[f"{key}_wins"] = f"{x[\'b_wins_per_folds\']}/{x[\'n_folds\']}"\n        c[f"{key}_verdict"] = x["verdict"]\n\n    d("1. boosting library", "xgb_vs_lgbm")\n    d("2. imbalance: class_weight", "cw")\n    d("2. imbalance: undersample", "us")\n    d("2. imbalance: smote", "smote")\n    d("3. tuning", "tuning")\n    d("4. boosting vs logistic", "boost_vs_lr")\n    d("5. ablation (reported only)", "ai_flag")\n    summ = _csv("p2_model_summary").set_index("pipeline")\n    best_boost = dec["4. boosting vs logistic"]["b"]\n    c |= {\n        "best_boost_name": best_boost,\n        "best_boost_pr_auc": f3(summ.loc[best_boost, "pr_auc_mean"]),\n        "lgbm_pr_auc": f3(summ.loc["lgbm", "pr_auc_mean"]),\n        "smote_pr_auc": f3(summ.loc["lgbm_smote", "pr_auc_mean"]),\n        "lr_balanced_brier": f"{summ.loc[\'logreg_balanced\', \'brier_mean\']:.3f}",\n        "lr_brier": f"{summ.loc[\'logreg\', \'brier_mean\']:.3f}",\n    }\n    trials = _csv("p2_tuning_trials")\n    top5 = trials.head(5)["num_leaves"]\n    c |= {\n        "tuning_top5_leaves": ", ".join(str(int(v)) for v in top5),\n        "tuning_n_trials": str(len(trials)),\n    }\n    sd = _csv("p2_smote_diagnostic").set_index("feature")\n    flags = sd.loc[[f for f in C.BOOLEAN_COLS if f in sd.index]]\n    c |= {\n        "smote_flag_ratio": f"{(flags[\'synthetic_fraud_mean\'] / flags[\'real_fraud_mean\']).mean():.2f}",\n        "smote_ipmis_real": pct(sd.loc["ip_country_mismatch", "real_fraud_mean"], 0),\n        "smote_ipmis_syn": pct(sd.loc["ip_country_mismatch", "synthetic_fraud_mean"], 0),\n        "smote_foreign_real": pct(sd.loc["is_foreign_transaction", "real_fraud_mean"], 0),\n        "smote_foreign_syn": pct(sd.loc["is_foreign_transaction", "synthetic_fraud_mean"], 0),\n    }\n    cal = [x for k, x in dec.items() if k.startswith("6.")]\n    c["calibration_verdict"] = (\n        "neither sigmoid nor isotonic calibration improved the Brier score"\n        if all(x["verdict"] == "not distinguishable" for x in cal)\n        else "calibration changed the Brier score (see notebook 04)"\n    )\n\n    thr = _j("p2_threshold")\n    cp = thr["cost_params"]\n    c |= {\n        "threshold": f"{thr[\'threshold\']:.4f}",\n        "thr_repeat_range": f"{min(thr[\'best_threshold_per_repeat\']):.4f}–{max(thr[\'best_threshold_per_repeat\']):.4f}",\n        "cv_recall": pct(thr["recall"]),\n        "cv_precision": pct(thr["precision"]),\n        "cv_alert_rate": pct(thr["alert_rate"], 2),\n        "cv_cost_per_1000": usd(thr["cost_per_1000_txn"]),\n        "cv_nothing_per_1000": usd(thr["flag_nothing_cost_usd"] / thr["n_rows"] * 1000),\n        "cv_saving": pct(thr["saving_vs_flag_nothing"]),\n        "alerts_per_fraud": f"{1 / thr[\'precision\']:.1f}",\n        "cost_review": f"{cp[\'review_cost_per_alert\']:g}",\n        "cost_friction": f"{cp[\'fp_friction_cost\']:g}",\n        "cost_fn_frac": f"{cp[\'fn_loss_fraction\']:g}",\n        "cost_fn_fixed": f"{cp[\'fn_fixed_cost\']:g}",\n        "cost_label": thr["cost_label"],\n    }\n    sens = _csv("p2_cost_sensitivity")\n    sn = sens[sens["parameter"] != "(base)"]\n    w = sn.loc[sn["regret_per_1000_txn"].idxmax()]\n    c |= {\n        "sens_thr_min": f"{sn[\'best_threshold\'].min():.4f}",\n        "sens_thr_max": f"{sn[\'best_threshold\'].max():.4f}",\n        "sens_worst_param": f"{w[\'parameter\']} ×{w[\'factor\']:g}",\n        "sens_worst_regret": usd(w["regret_per_1000_txn"]),\n    }\n\n    tt = _j("p2_three_tier")\n    c |= {\n        "tt_stepup": f"{tt[\'t_stepup\']:.4f}",\n        "tt_decline": f"{tt[\'t_decline\']:.4f}",\n        "tt_pass_rate": f"{tt[\'params\'][\'stepup_fraud_pass_rate\']:g}",\n        "tt_pass_rate_pct": pct(tt["params"]["stepup_fraud_pass_rate"], 0),\n        "tt_decline_fp": f"{tt[\'params\'][\'decline_fp_cost\']:g}",\n        "tt_decline_tp": f"{tt[\'params\'][\'decline_tp_cost\']:g}",\n        "tt_cv_diff": f"{tt[\'cost_per_1000_txn\'] - tt[\'two_tier_same_model_cost_per_1000\']:+,.0f} USD",\n    }\n\n    h = _j("p2_holdout")\n    m = h["metrics"]\n    post = _j("p2_holdout_posthoc")\n    sv = post["two_tier_saving_vs_nothing"]\n    t3 = post["three_tier_model__three_minus_two_per_1000"]\n    c |= {\n        "ho_n": f"{h[\'n_rows\']:,}",\n        "ho_fraud": f"{h[\'n_fraud\']}",\n        "ho_pr_auc": f3(m["pr_auc"]["value"]),\n        "ho_pr_auc_ci": ci3(m["pr_auc"]["ci_low"], m["pr_auc"]["ci_high"]),\n        "ho_roc_auc": f3(m["roc_auc"]["value"]),\n        "ho_roc_auc_ci": ci3(m["roc_auc"]["ci_low"], m["roc_auc"]["ci_high"]),\n        "ho_noskill": f"{m[\'base_rate\'][\'value\']:.4f}",\n        "ho_recall": pct(m["recall"]["value"]),\n        "ho_recall_ci": ci_pct(m["recall"]["ci_low"], m["recall"]["ci_high"]),\n        "ho_precision": pct(m["precision"]["value"]),\n        "ho_precision_ci": ci_pct(m["precision"]["ci_low"], m["precision"]["ci_high"]),\n        "ho_alert_rate": pct(m["alert_rate"]["value"], 2),\n        "ho_alert_rate_ci": ci_pct(m["alert_rate"]["ci_low"], m["alert_rate"]["ci_high"], 2),\n        "ho_saving": pct(sv["value"]),\n        "ho_saving_ci": ci_pct(sv["ci_low"], sv["ci_high"]),\n        "ho_tt_diff": f"{t3[\'value\']:+,.0f} USD",\n        "ho_tt_diff_ci": f"[{t3[\'ci_low\']:+,.0f}, {t3[\'ci_high\']:+,.0f}]",\n    }\n\n    shap = _j("p2_shap")\n    gi = shap["global_importance"]\n    c["shap_top3"] = ", ".join(f"`{x[\'feature\']}`" for x in gi[:3])\n    c["shap_top3_share"] = pct(\n        sum(x["mean_abs_shap"] for x in gi[:3]) / sum(x["mean_abs_shap"] for x in gi), 0\n    )\n    ex = {e["label"]: e for e in shap["examples"]}\n    c["shap_missed_amount"] = f"{ex[\'missed fraud\'][\'amount_usd\']:,.2f} USD"\n    return c\n\n\n_PLACEHOLDER = re.compile(r"\\{\\{\\s*(\\w+)\\s*\\}\\}")\n\n\ndef render(template: str, ctx: dict[str, str] | None = None) -> str:\n    ctx = build_context() if ctx is None else ctx\n    missing = sorted({k for k in _PLACEHOLDER.findall(template) if k not in ctx})\n    if missing:\n        raise KeyError(f"template uses unknown keys: {missing}")\n    return _PLACEHOLDER.sub(lambda mt: ctx[mt.group(1)], template)\n\n\ndef render_file(template_path: Path, out_path: Path, header: str = "") -> str:\n    text = header + render(template_path.read_text())\n    out_path.write_text(text)\n    return text\n', 'src/fraud_e2e/threshold.py': '"""Threshold selection by expected cost on out-of-fold (OOF) predictions.\n\nThe OOF frame has one row per (repeat, transaction) with columns\nrepeat, y, amount_usd, score. Each repeat is a full set of dev predictions, so\neach gives its own cost curve; curves are averaged across repeats and the\nminimum is the chosen threshold (pre-registration A2).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e.cost import (\n    CostParams,\n    ThreeTierParams,\n    confusion_by_threshold,\n    cost_from_confusion,\n    require_usable,\n    scaled,\n)\n\nNEAR_OPTIMAL_TOL = 0.01  # thresholds within 1% of the minimum cost are "near-optimal"\n\n\ndef threshold_grid() -> np.ndarray:\n    """Fine grid for 1-D optimisation: dense near 0 (rare-class thresholds), covers [0, 1]."""\n    return np.unique(np.r_[0.0, np.geomspace(1e-4, 1.0, 4001), np.linspace(0, 1, 1001)])\n\n\ndef web_grid(extra=()) -> np.ndarray:\n    """Coarser grid for the website export and the 2-D three-tier search, plus extra points."""\n    return np.unique(np.r_[0.0, np.geomspace(1e-4, 1.0, 481), np.linspace(0, 1, 201), extra])\n\n\ndef near_optimal_range(grid: np.ndarray, mean_curve: np.ndarray, tol=NEAR_OPTIMAL_TOL):\n    """Lowest and highest threshold whose mean cost is within `tol` of the minimum."""\n    ok = mean_curve <= mean_curve.min() * (1 + tol)\n    return float(grid[ok].min()), float(grid[ok].max())\n\n\ndef confusion_per_repeat(oof: pd.DataFrame, grid: np.ndarray) -> dict[int, dict]:\n    return {\n        r: confusion_by_threshold(g["y"], g["score"], g["amount_usd"], grid)\n        for r, g in oof.groupby("repeat")\n    }\n\n\ndef cost_curves(conf: dict[int, dict], p: CostParams) -> np.ndarray:\n    """Array (n_repeats, n_thresholds) of total cost in USD."""\n    return np.vstack([cost_from_confusion(c, p) for c in conf.values()])\n\n\ndef operating_point(conf: dict[int, dict], i: int, n_rows: int) -> dict:\n    """Mean over repeats of the operating-point statistics at grid index i."""\n    tp = np.mean([c["tp"][i] for c in conf.values()])\n    fp = np.mean([c["fp"][i] for c in conf.values()])\n    fn = np.mean([c["fn"][i] for c in conf.values()])\n    return {\n        "recall": tp / (tp + fn),\n        "precision": tp / (tp + fp) if tp + fp > 0 else float("nan"),\n        "alert_rate": (tp + fp) / n_rows,\n        "tp": tp,\n        "fp": fp,\n        "fn": fn,\n    }\n\n\ndef choose_threshold(oof: pd.DataFrame, p: CostParams, grid: np.ndarray | None = None) -> dict:\n    require_usable(p)\n    grid = threshold_grid() if grid is None else grid\n    conf = confusion_per_repeat(oof, grid)\n    curves = cost_curves(conf, p)\n    mean_curve = curves.mean(axis=0)\n    i = int(np.argmin(mean_curve))\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    per_repeat_best = [float(grid[np.argmin(c)]) for c in curves]\n    y = oof[oof["repeat"] == oof["repeat"].iloc[0]]\n    nothing = float(\n        (p.fn_loss_fraction * y.loc[y["y"] == 1, "amount_usd"]).sum()\n        + p.fn_fixed_cost * y["y"].sum()\n    )\n    return {\n        "threshold": float(grid[i]),\n        "grid_index": i,\n        "cost_usd": float(mean_curve[i]),\n        "cost_per_1000_txn": float(mean_curve[i] / n_rows * 1000),\n        "cost_sd_across_repeats": float(curves[:, i].std(ddof=1)) if len(curves) > 1 else None,\n        "best_threshold_per_repeat": per_repeat_best,\n        "near_optimal_range": near_optimal_range(grid, mean_curve),\n        "flag_nothing_cost_usd": nothing,\n        "saving_vs_flag_nothing": float(1 - mean_curve[i] / nothing),\n        "n_rows": n_rows,\n        **operating_point(conf, i, n_rows),\n        "_curves": curves,\n        "_grid": grid,\n    }\n\n\ndef sensitivity(oof: pd.DataFrame, base: CostParams, factors=(0.5, 2.0)) -> pd.DataFrame:\n    """Scale one parameter at a time; re-optimise the threshold; also cost of keeping base."""\n    grid = threshold_grid()\n    conf = confusion_per_repeat(oof, grid)\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    base_i = int(np.argmin(cost_curves(conf, base).mean(axis=0)))\n    rows = [{"parameter": "(base)", "factor": 1.0, "value": None}]\n    for name in base.numeric():\n        for f in factors:\n            rows.append({"parameter": name, "factor": f, "value": getattr(base, name) * f})\n    out = []\n    for r in rows:\n        p = base if r["parameter"] == "(base)" else scaled(base, r["parameter"], r["factor"])\n        mean_curve = cost_curves(conf, p).mean(axis=0)\n        i = int(np.argmin(mean_curve))\n        lo, hi = near_optimal_range(grid, mean_curve)\n        out.append(\n            {\n                **r,\n                "best_threshold": float(grid[i]),\n                "near_optimal_low": lo,\n                "near_optimal_high": hi,\n                "cost_per_1000_txn": float(mean_curve[i] / n_rows * 1000),\n                "cost_per_1000_at_base_threshold": float(mean_curve[base_i] / n_rows * 1000),\n                "regret_per_1000_txn": float((mean_curve[base_i] - mean_curve[i]) / n_rows * 1000),\n                **{\n                    k: v\n                    for k, v in operating_point(conf, i, n_rows).items()\n                    if k in ("recall", "precision", "alert_rate")\n                },\n            }\n        )\n    return pd.DataFrame(out)\n\n\ndef export_for_web(oof: pd.DataFrame, grid: np.ndarray | None = None) -> dict:\n    """Per-threshold sufficient statistics (mean over repeats) for the static website."""\n    grid = web_grid() if grid is None else grid\n    conf = confusion_per_repeat(oof, grid)\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    keys = ["tp", "fp", "fn", "tn", "fn_amount_usd"]\n    mean = {k: np.mean([c[k] for c in conf.values()], axis=0) for k in keys}\n    return {\n        "n_rows": n_rows,\n        "n_fraud": int(mean["tp"][0] + mean["fn"][0]),\n        "fraud_amount_usd": float(\n            oof.loc[oof["y"] == 1].groupby("repeat")["amount_usd"].sum().mean()\n        ),\n        "threshold": grid.round(6).tolist(),\n        **{k: np.round(v, 3).tolist() for k, v in mean.items()},\n        "cost_formula": (\n            "(tp + fp) * review_cost_per_alert + fp * fp_friction_cost"\n            " + fn_loss_fraction * fn_amount_usd + fn * fn_fixed_cost"\n        ),\n        "note": "Mean over 3 CV repeats of out-of-fold predictions on the dev split.",\n    }\n\n\n# --------------------------------------------------------------------------- three-tier\n\n\ndef _prefix_at_grid(y, s, a, grid, p: CostParams):\n    order = np.argsort(s)\n    ys, as_ = y[order].astype(bool), a[order]\n    idx = np.searchsorted(s[order], grid, side="left")  # rows with score < t\n    fn_cost = np.where(ys, p.fn_loss_fraction * as_ + p.fn_fixed_cost, 0.0)\n    pre = lambda v: np.r_[0.0, np.cumsum(v)]  # noqa: E731\n    return {\n        "N": pre(np.ones(len(ys)))[idx],\n        "L": pre(~ys)[idx],\n        "F": pre(ys)[idx],\n        "FNC": pre(fn_cost)[idx],\n        "tot": {\n            "N": len(ys),\n            "L": float((~ys).sum()),\n            "F": float(ys.sum()),\n            "FNC": float(fn_cost.sum()),\n        },\n    }\n\n\ndef three_tier_cost_matrix(y, score, amount, grid, p: CostParams, q: ThreeTierParams):\n    """cost[i, j] for t_stepup = grid[i], t_decline = grid[j] (inf where j < i).\n\n    approve: score < t1; step-up: t1 <= score < t2; decline: score >= t2.\n    """\n    P = _prefix_at_grid(np.asarray(y), np.asarray(score, float), np.asarray(amount, float), grid, p)\n    t = P["tot"]\n    N, L, F, FNC = P["N"], P["L"], P["F"], P["FNC"]\n    approve = FNC[:, None]\n    stepup = (\n        p.review_cost_per_alert * (N[None, :] - N[:, None])\n        + p.fp_friction_cost * (L[None, :] - L[:, None])\n        + q.stepup_fraud_pass_rate * (FNC[None, :] - FNC[:, None])\n    )\n    decline = q.decline_fp_cost * (t["L"] - L[None, :]) + q.decline_tp_cost * (t["F"] - F[None, :])\n    cost = approve + stepup + decline\n    cost[np.tril_indices(len(grid), -1)] = np.inf\n    return cost\n\n\ndef choose_three_tier(\n    oof: pd.DataFrame, p: CostParams, q: ThreeTierParams, grid: np.ndarray | None = None\n) -> dict:\n    require_usable(p, q)\n    grid = web_grid() if grid is None else grid\n    grid = np.r_[grid, np.inf]  # last column = "never decline" (two-tier)\n    mats = [\n        three_tier_cost_matrix(g["y"].to_numpy(), g["score"], g["amount_usd"], grid, p, q)\n        for _, g in oof.groupby("repeat")\n    ]\n    mean = np.mean(mats, axis=0)\n    i, j = np.unravel_index(np.argmin(mean), mean.shape)\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    # two-tier under the same three-tier cost model = never decline (t2 beyond max score)\n    two_tier_i = int(np.argmin(mean[:, -1]))\n    return {\n        "t_stepup": float(grid[i]),\n        "t_decline": float(grid[j]),\n        "cost_per_1000_txn": float(mean[i, j] / n_rows * 1000),\n        "two_tier_same_model_threshold": float(grid[two_tier_i]),\n        "two_tier_same_model_cost_per_1000": float(mean[two_tier_i, -1] / n_rows * 1000),\n        "cost_sd_across_repeats_per_1000": float(\n            np.std([m[i, j] for m in mats], ddof=1) / n_rows * 1000\n        ),\n        "best_pair_per_repeat": [\n            [float(grid[a]), float(grid[b])]\n            for a, b in (np.unravel_index(np.argmin(m), m.shape) for m in mats)\n        ],\n        "n_rows": n_rows,\n    }\n', 'configs/cost.yaml': '# Cost assumptions for threshold selection. See notes/01_cost_assumptions.md.\n# ILLUSTRATIVE VALUES — assumed for this portfolio project, NOT sourced from a bank.\nstatus: accepted          # accepted by Nat on 2026-11-09 as illustrative\nlabel: "Illustrative assumed costs (not bank data)"\nparams:\n  review_cost_per_alert: 5.0   # USD per step-up / review (every flagged transaction)\n  fp_friction_cost: 10.0       # USD extra per good customer sent to step-up\n  fn_loss_fraction: 1.0        # multiplier on amount_usd lost on a missed fraud\n  fn_fixed_cost: 15.0          # USD per missed fraud (chargeback handling)\n# Sensitivity analysis: each parameter scaled by these factors, others at base.\nsensitivity_factors: [0.5, 2.0]\n\n# Supplementary three-tier policy (approve / step-up / decline).\n# status must be "accepted" before these are used. See notes/01_cost_assumptions.md.\nthree_tier:\n  status: accepted        # accepted by Nat on 2026-11-09 as illustrative\n  params:\n    stepup_fraud_pass_rate: 0.2   # share of fraud that still gets through step-up\n    decline_fp_cost: 50.0         # USD per good customer hard-declined\n    decline_tp_cost: 0.0          # USD per fraud declined (automatic, no review)\n'}
BINARY = {}
for rel, text in FILES.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
for rel, b64 in BINARY.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
sys.path.insert(0, str(WORK / "src"))
from fraud_e2e import config as C
from fraud_e2e.data import load_raw
from fraud_e2e.splits import make_splits, save_splits
if not C.SPLITS_CSV.exists():
    save_splits(make_splits(load_raw()))
print("fraud_e2e ready; data:", C.RAW_CSV)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from fraud_e2e import config as C
from fraud_e2e import plots as P
from fraud_e2e.cv import metric_ci, pick, run_cv, save_result, verdict_text
from fraud_e2e.data import load_raw
from fraud_e2e.features import feature_frame
from fraud_e2e.models import fit_pipeline
from fraud_e2e.reporting import write_json
from fraud_e2e.splits import dev_only

P.set_style()
dev = dev_only(load_raw())
print(f"dev: {len(dev):,} rows, {int(dev[C.TARGET].sum())} fraud")

## 1. Cross-validate both variants

In [ ]:
lr = run_cv(dev, {"model": "logreg"}, "logreg")
lr_bal = run_cv(dev, {"model": "logreg", "imbalance": "class_weight"}, "logreg_balanced")
for r in (lr, lr_bal):
    save_result(r)


def table(results):
    rows = []
    for r in results:
        row = {"pipeline": r.name}
        for m in ["pr_auc", "roc_auc", "brier"]:
            ci = metric_ci(r, m)
            row[m] = f"{ci['mean']:.3f} [{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]"
            row[f"{m}_sd"] = round(ci["sd"], 3)
        rows.append(row)
    return pd.DataFrame(rows)


table([lr, lr_bal])

Values are the mean over 15 folds with a corrected 95% interval. The no-skill PR-AUC equals the
fraud rate of a fold, about 0.017.

In [ ]:
fig, axes = P.plt.subplots(1, 2, figsize=(13, 3.2))
P.fold_dots([lr, lr_bal], "pr_auc", ax=axes[0], xlabel="PR-AUC per fold (dots); mean ± 95% CI")
P.fold_dots([lr, lr_bal], "roc_auc", ax=axes[1], xlabel="ROC-AUC per fold (dots); mean ± 95% CI")
axes[0].set_title("PR-AUC (primary)")
axes[1].set_title("ROC-AUC")
fig.tight_layout()
P.save(fig, "p2_baseline_folds")
P.plt.show()

## 2. Paired comparison (pre-registered rule)

In [ ]:
chosen, test_ap = pick(lr, lr_bal, "pr_auc")
_, test_roc = pick(lr, lr_bal, "roc_auc")
_, test_brier = pick(lr, lr_bal, "brier", lower_is_better=True)
cmp = pd.DataFrame([test_ap, test_roc, test_brier])[
    ["metric", "mean_diff_b_minus_a", "ci_low", "ci_high", "p_value", "b_wins_per_folds", "verdict"]
]
cmp

In [ ]:
display(
    Markdown(
        f"PR-AUC difference (balanced − unweighted): **{test_ap['mean_diff_b_minus_a']:+.4f}** "
        f"(95% CI {test_ap['ci_low']:+.4f} to {test_ap['ci_high']:+.4f}), balanced better in "
        f"{test_ap['b_wins_per_folds']} of {test_ap['n_folds']} folds → verdict: "
        f"**{verdict_text(test_ap)}**. Baseline carried forward: **`{chosen.name}`**."
    )
)

Class weighting reweights the loss; it is expected to change probabilities (Brier) far more than
ranking (PR-AUC), which is why the decision is made on PR-AUC.

In [ ]:
fig, ax = P.plt.subplots(figsize=(5.5, 4))
P.pr_curves([lr, lr_bal], ax=ax)
ax.set_title("Precision-recall, out-of-fold (repeat 0)")
P.save(fig, "p2_baseline_pr")
P.plt.show()

## 3. What the baseline relies on

Coefficients of the chosen variant refit on all dev rows. Inputs are standardised (or 0/1 for
booleans and one-hot levels), so a coefficient is the change in log-odds per standard deviation
(or for the flag/level being present).

In [ ]:
model = fit_pipeline(chosen.spec, feature_frame(dev), dev[C.TARGET].to_numpy())
names = model.named_steps["prep"].get_feature_names_out()
coef = pd.DataFrame({"input": names, "coef": model.named_steps["model"].coef_[0]})
coef["input"] = coef["input"].str.replace(r"^\w+?__", "", regex=True)
top = coef.reindex(coef["coef"].abs().sort_values(ascending=False).index).head(15)

fig, ax = P.plt.subplots(figsize=(6.5, 5))
t = top.iloc[::-1]
ax.barh(t["input"], t["coef"], color=[P.FRAUD if c > 0 else P.LEGIT for c in t["coef"]], height=0.6)
ax.axvline(0, color=P.NEUTRAL, lw=1)
ax.set_xlabel("Coefficient (log-odds per SD, or per flag/level present)")
ax.set_title(f"Top 15 coefficients — {chosen.name}")
ax.grid(axis="y", visible=False)
P.save(fig, "p2_baseline_coefficients")
P.plt.show()

## 4. Save

In [ ]:
write_json(
    {
        "pipelines": {
            r.name: {m: metric_ci(r, m) for m in ["pr_auc", "roc_auc", "brier"]} | {"spec": r.spec}
            for r in (lr, lr_bal)
        },
        "comparisons": [test_ap, test_roc, test_brier],
        "chosen_baseline": chosen.name,
        "top_coefficients": top.to_dict(orient="records"),
    },
    "p2_baseline",
)

## Summary

In [ ]:
ci = metric_ci(chosen, "pr_auc")
roc = metric_ci(chosen, "roc_auc")
base = chosen.folds["base_rate"].mean()
pos = top[top["coef"] > 0].head(3)["input"].tolist()
display(
    Markdown(f"""
- **Baseline:** `{chosen.name}` — PR-AUC **{ci["mean"]:.3f}** (95% CI {ci["ci_low"]:.3f}–{ci["ci_high"]:.3f},
  fold SD {ci["sd"]:.3f}), about **{ci["mean"] / base:.0f}×** the no-skill value of {base:.3f};
  ROC-AUC {roc["mean"]:.3f} ({roc["ci_low"]:.3f}–{roc["ci_high"]:.3f}).
- **Class weighting:** on PR-AUC, {verdict_text(test_ap)} (difference {test_ap["mean_diff_b_minus_a"]:+.4f},
  CI {test_ap["ci_low"]:+.4f} to {test_ap["ci_high"]:+.4f}), so the rule keeps `{chosen.name}`.
- **Strongest positive drivers:** {", ".join(f"`{p}`" for p in pos)}.
- **Fold-to-fold spread:** the PR-AUC SD across folds ({ci["sd"]:.3f}) is
  {ci["sd"] / abs(test_ap["mean_diff_b_minus_a"]):.1f}× the size of the weighting effect — the
  yardstick for every comparison in notebook 04.
""")
)